# ENRICH-Aerial `nadir-2` — GCP bundle adjustment with **scop**, built by hand

Same block, same observations and same sigmas as
[`ENRICH-Aerial_pycolmap_sfm_gcp.ipynb`](ENRICH-Aerial_pycolmap_sfm_gcp.ipynb),
but the adjustment runs in **scop**'s rigorous least-squares engine instead of
COLMAP's Ceres problem. scop is not an SfM engine, so the input is the *rough
free network* produced by pycolmap; scop then does the full bundle with camera
position priors, control points and held-out check points.

**Everything is assembled by hand.** `scop.utils.loaders.load_colmap_reconstruction`
would do the whole import in one line, but the point of this notebook is to learn
the scop object model, so every `Camera`, `Image`, `Point3D` and `Point2D` is
created explicitly and pushed into the `Optimizer` with `add_object`. The loader
([`scop/utils/loaders/colmap.py`](file:///home/francesco/dev/scop/scop/utils/loaders/colmap.py))
stays the reference for the COLMAP↔scop conventions, and can be swapped in later.

The `database.db` is **not** needed: the reconstruction already carries every 2D
observation (`image.points2D`, each with `xy` and `point3D_id`). Raw keypoints and
verified matches (`scop/utils/loaders/colmap_db.py`) only matter if you want to
re-triangulate from scratch.

> `opt` is stateful — re-run from the *Assemble the Optimizer* cell downwards,
> never a single fit cell in isolation.


## 0. Imports, paths and switches

`MIN_TRACK_LENGTH` / `MAX_POINTS` subsample the tie-point network. The default
(3 / 5000 → ~15 k variables, ~0.4 s per LM iteration) keeps every cell snappy
while you experiment; `2 / None` loads the full block (43 749 points,
131 566 variables, ~2 s per iteration, ~1 min per fit cell).


In [ ]:
import math
import time
from pathlib import Path

import numpy as np
import pycolmap
from scop.utils.core.Camera import Camera, ModelType
from scop.utils.core.Image import Image, Point2D
from scop.utils.core.Optimizer import Optimizer
from scop.utils.core.Point3D import Point3D
from scop.utils.geometry.Rotation import opk_to_rmat, rmat_to_opk

import enrich_data

ROOT = Path(enrich_data.__file__).parents[1]
SPARSE = ROOT / "data/ENRICH-Aerial/nadir-2/scop/colmap_sparse_free"

MIN_TRACK_LENGTH = 3
MAX_POINTS = None  # 5000; None: full block
REFINE_FOCAL = True  # free the focal length (self-calibration)
USE_CAMERA_PRIORS = True  # add the GNSS-like camera position priors

# COLMAP -> scop camera frame: a 180 deg rotation about x.
FLIP = np.diag([1.0, -1.0, -1.0])

# Id ranges, kept disjoint from the COLMAP ids (tie point ids stop at 45 571).
GCP_ID0 = 10_000_000
MARK_ID0 = 30_000_000


## 1. Data — ground truth and the (optionally perturbed) survey


In [24]:
# `load()` reads the CSVs as they are; `add_noise()` returns another object of
# the same type with a simulated survey applied. `truth` is kept for scoring.

truth = enrich_data.load(subblock="nadir-2", split_control_check=True)
data = enrich_data.load(subblock="nadir-2", split_control_check=True)
# data = enrich_data.add_noise(truth)   # <- flip to run against a noisy survey

print(enrich_data.summary(data))
print(
    f"survey noise: xy {enrich_data.gcp_sigma_m[0] * 100:.0f} cm, "
    f"z {enrich_data.gcp_sigma_m[2] * 100:.0f} cm; marks {enrich_data.gcp_sigma_px} px"
)
print(f"GCPs: {data.control_names}")
print(f"CP:   {data.check_names}")


156 marker image marks, 26 markers, 39 images
control: 18 points | check: 8 points (gcp.001, gcp.004, gcp.010, gcp.013, gcp.016, gcp.019, gcp.022, gcp.025)
survey noise: xy 2 cm, z 3 cm; marks 1.0 px
GCPs: ['gcp.002', 'gcp.003', 'gcp.005', 'gcp.006', 'gcp.007', 'gcp.008', 'gcp.009', 'gcp.011', 'gcp.012', 'gcp.014', 'gcp.015', 'gcp.017', 'gcp.018', 'gcp.020', 'gcp.021', 'gcp.023', 'gcp.024', 'gcp.026']
CP:   ['gcp.001', 'gcp.004', 'gcp.010', 'gcp.013', 'gcp.016', 'gcp.019', 'gcp.022', 'gcp.025']


## 2. The free network

The reconstruction in `scop/colmap_sparse_free` has already been Sim3-aligned onto
the camera positions (centre RMSE 1.5 cm). That step matters: a similarity leaves
the reprojection residuals untouched, but scop's BA is a *local* optimiser, so the
block has to sit in the world frame before the GCP and position constraints mean
anything. If the assert below ever fires, re-run the alignment upstream.


In [25]:
rec = pycolmap.Reconstruction(SPARSE)

d = np.array([
    im.cam_from_world().inverse().translation - truth.camera_xyz[im.name]
    for im in rec.images.values()
])
centre_rmse = float(np.sqrt((d**2).sum(1).mean()))

print(
    f"{rec.num_images()} images, {rec.num_points3D()} points3D, "
    f"{rec.num_cameras()} camera"
)
print(f"mean reprojection error : {rec.compute_mean_reprojection_error():.3f} px")
print(f"camera-centre RMSE      : {centre_rmse:.4f} m")
print(rec.cameras[1])

assert centre_rmse < 1.0, "free network is not in the world frame - align it first"

39 images, 43749 points3D, 1 camera
mean reprojection error : 0.607 px
camera-centre RMSE      : 0.0155 m
Camera(camera_id=1, model=SIMPLE_PINHOLE, width=6016, height=4016, params=[5865.18, 3008, 2008] (f, cx, cy))


## 3. The `Camera`

COLMAP's `SIMPLE_PINHOLE` (`f, cx, cy`) maps to a scop **Frame** camera with
`ModelType.SimpleRadial` and `k = 0` — that is exactly what the scop loader does
(`colmap.py:319`, *"SimpleRadial with k=0 is pinhole"*).

Two things to get right:

- The distortion parameter of `SimpleRadial` is named **`k`**, not `k1`
  (`Model.py:427`; the projection `f_simple_radial` reads `_s['k']`). Passing
  `k1=` silently creates an unused attribute — the scop loader does exactly that
  at `colmap.py:229`, which is harmless here (`k = 0`) but a real bug for
  `SIMPLE_RADIAL` models.
- Only **`f`** may be freed. scop's SimpleRadial projects as
  `v = h - ((y + dy)·f + cy)`, so refining `cy` moves the principal point in the
  *opposite* direction to COLMAP; and `k` must stay pinned at 0 for parity.
  Never re-fix a parameter after freeing it either — the jitted constant goes
  stale (`Image.py:829`); rebuild the camera instead.


In [26]:
ccam = rec.cameras[1]
f0, cx0, cy0 = (float(v) for v in ccam.params)

cam = Camera.frame(
    width=ccam.width,
    height=ccam.height,
    focal_px=f0,
    model=ModelType.SimpleRadial,
    cx=cx0,
    cy=cy0,
    k=0.0,
    variables=["f"] if REFINE_FOCAL else None,  # None = every intrinsic fixed
)
cam._id = 1

print(f"Camera {cam.w:.0f}x{cam.h:.0f}, f={cam.f:.4f}, cx={cam.cx}, cy={cam.cy}")
print(f"free intrinsics: {list(cam.get_variables())}")


Camera 6016x4016, f=5865.1811, cx=3008.0, cy=2008.0
free intrinsics: ['f']


## 4. The `Image`s — pose conversion and position priors

COLMAP stores `cam_from_world` as $(R, t)$ with $+Z$ forward and $+Y$ down; scop
uses a camera looking down $-Z$ with $+Y$ up, and parameterises the pose as the
camera centre $S$ plus omega/phi/kappa (radians). The conversion, straight from
`colmap.py:459-484`:

$$R_{\text{scop}} = \operatorname{diag}(1,-1,-1)\,R_{\text{colmap}}, \qquad
S = -R_{\text{colmap}}^{\top} t, \qquad
(\omega,\varphi,\kappa) = \texttt{rmat\_to\_opk}(R_{\text{scop}})$$

### ⚠️ The `observations` trap

`Image.at` assigns its `observations` argument **verbatim** (`Image.py:1224`) —
it does not merge. The default is `["Point2D"]`, a sentinel that switches on the
reprojection residuals. So passing `observations=["Xs", "Ys", "Zs"]` to add GNSS
priors *silently removes every image residual* and leaves you adjusting position
priors against nothing. The sentinel must be listed explicitly:

```python
observations=["Xs", "Ys", "Zs", "Point2D"]
```

`img.check_observations([...])` fails the same way — it filters `"Point2D"` out
because it is not a dataclass field (`Observation.py:474`). The assert at the
bottom of the cell guards against both.

Note also that we overwrite `img.Xs/Ys/Zs` with the *surveyed* centres (the
observed value) while leaving the estimated values at the COLMAP pose — that is
what makes them priors rather than a reset of the network.


In [27]:
obs_names = ["Xs", "Ys", "Zs", "Point2D"] if USE_CAMERA_PRIORS else ["Point2D"]

images, id2name, name2id = {}, {}, {}
for im in rec.images.values():
    cfw = im.cam_from_world()
    R_scop = FLIP @ cfw.rotation.matrix()
    S = cfw.inverse().translation  # camera centre
    omega, phi, kappa = rmat_to_opk(R_scop, np, np)  # radians

    img = Image.at(
        camera=cam,
        x=float(S[0]),
        y=float(S[1]),
        z=float(S[2]),
        omega=float(omega),
        phi=float(phi),
        kappa=float(kappa),
        variables=["Xs", "Ys", "Zs", "omega", "phi", "kappa"],
        observations=list(obs_names),
        _id=im.image_id,
    )

    if USE_CAMERA_PRIORS:
        # Observed value only - the estimated pose stays at the COLMAP solution.
        # float() because enrich_data returns float32 arrays.
        img.Xs, img.Ys, img.Zs = (float(v) for v in data.camera_xyz[im.name])
        for axis in ("Xs", "Ys", "Zs"):
            img.set_sigma_var(axis, enrich_data.camera_position_sigma_m)

    images[im.image_id] = img
    id2name[im.image_id] = im.name
    name2id[im.name] = im.image_id

assert all("Point2D" in i._observations for i in images.values()), (
    "the Point2D sentinel was dropped - the images have no reprojection residuals"
)

print(f"{len(images)} images, observations = {images[min(images)]._observations}")
print(f"free EO parameters per image: {list(images[min(images)].get_variables())}")


39 images, observations = ['Xs', 'Ys', 'Zs', 'Point2D']
free EO parameters per image: ['Xs', 'Ys', 'Zs', 'omega', 'phi', 'kappa']


## 5. Tie points and their image marks

One `Point3D.tie` per kept COLMAP point (ids preserved), one `Point2D` per
observation. `Point2D` carries a reference to its 3D point (`_m_pt3`) and is
attached to the image with `add_pt2`; it produces no residual of its own — the
two reprojection rows come from the `PointProjectionProxy` that `Image` builds
around it (`Image.py:212`).

Three details worth knowing:

- **No y-flip.** Raw full-res, top-left `u, v` go in as they are. scop's
  SimpleRadial ends with `v = h - (...)`, which exactly cancels the y-inversion
  introduced by `R_scop = diag(1,-1,-1) · R_colmap` (`colmap.py:506-531`).
- **The id scheme collides.** `img_id·100000 + pt3_id` overwrites entries when the
  same `(image, point3D)` pair appears twice, so `len(opt.dict_obj["Point2D"])`
  is *not* the observation count. Residuals come from `img._lst_pt2`, which is a
  list, so nothing is actually lost.
- **Drop points behind the camera.** scop disables them internally, which desyncs
  the batched Jacobian. Expect zero here — the network starts converged.


In [28]:
# Longest tracks first: the best-constrained points survive the subsampling.
pts_sorted = sorted(
    rec.points3D.items(), key=lambda kv: len(kv[1].track.elements), reverse=True
)

tie_points = {}
for pid, p in pts_sorted:
    if len(p.track.elements) < MIN_TRACK_LENGTH:
        continue
    if MAX_POINTS is not None and len(tie_points) >= MAX_POINTS:
        break
    tie_points[pid] = Point3D.tie(
        x=float(p.xyz[0]), y=float(p.xyz[1]), z=float(p.xyz[2]), _id=int(pid)
    )

In [29]:
lst_pt2, n_tie_obs, n_behind = [], 0, 0
for im in rec.images.values():
    img = images[im.image_id]
    cfw = im.cam_from_world()
    R_scop, S = FLIP @ cfw.rotation.matrix(), cfw.inverse().translation

    for p2 in im.points2D:
        pid = int(p2.point3D_id)
        if pid not in tie_points:
            continue
        pt3 = tie_points[pid]

        # scop's camera looks down -Z: a visible point has Zc < 0.
        if (R_scop @ (np.array([pt3.X, pt3.Y, pt3.Z]) - S))[2] >= 0:
            n_behind += 1
            continue

        pt2 = Point2D(
            _id=im.image_id * 100_000 + pid,
            x=float(p2.xy[0]),
            y=float(p2.xy[1]),  # raw u, v - no flip
            _m_pt3=pt3,
        )
        pt2.create_estimated()
        img.add_pt2(pt2)
        lst_pt2.append(pt2)
        n_tie_obs += 1

print(
    f"{len(tie_points)} tie points, {n_tie_obs} image marks "
    f"(min track {MIN_TRACK_LENGTH}, max {MAX_POINTS}), {n_behind} behind camera"
)

42635 tie points, 219432 image marks (min track 3, max None), 0 behind camera


## 6. Markers — control points, check points, and their marks

Three `Point3D` flavours matter here (`Point3D.py:77-104`):

| builder | `_variables` | `_observations` | meaning |
|---|---|---|---|
| `Point3D.control` | `X, Y, Z` | `X, Y, Z` | surveyed **and** adjusted → weighted prior, residual observable |
| `Point3D.check`   | `X, Y, Z` | *(none)*        | free, surveyed value kept only for scoring |
| `Point3D.tie`     | `X, Y, Z` | *(none)*        | ordinary tie point |

The control points get a **per-axis** sigma via `set_sigma_var`, so the looser
vertical survey accuracy is honoured — `Point3D.control(..., sigma=)` would apply
one isotropic value instead.

> **Deviation from the pycolmap run, on purpose.** scop resolves the pixel sigma
> at class level (`PointProjectionProxy.get_sigma_var`, `Image.py:252`), so a
> marker mark cannot carry its own weight. One sigma
> (`tie_points_sigma_px = 0.5 px`) therefore covers ties *and* marks, whereas the
> pycolmap notebook used 1.0 px on the marks. Public API only — no patching of
> `opt._block_sigmas`.


In [30]:
markers = {}
for i, name in enumerate(data.marker_names):
    x, y, z = (float(v) for v in data.marker_xyz[name])
    if name in data.control_names:
        pt3 = Point3D.control(x=x, y=y, z=z, _id=GCP_ID0 + i)
        for axis, sigma in zip("XYZ", enrich_data.gcp_sigma_m, strict=True):
            pt3.set_sigma_var(axis, float(sigma))  # per-axis, not isotropic
    else:
        pt3 = Point3D.check(x=x, y=y, z=z, _id=GCP_ID0 + i)
    markers[name] = pt3

n_marks = 0
for j, row in enumerate(data.marker_data.itertuples(index=False)):
    img_id = name2id.get(row.image_name)
    if img_id is None:  # image outside the sub-block
        continue
    pt2 = Point2D(
        _id=MARK_ID0 + j,
        x=float(row.u),
        y=float(row.v),
        _m_pt3=markers[row.marker_name],
    )
    pt2.create_estimated()
    images[img_id].add_pt2(pt2)
    lst_pt2.append(pt2)
    n_marks += 1

print(
    f"{len(markers)} markers = {len(data.control_names)} control "
    f"+ {len(data.check_names)} check, {n_marks} image marks"
)
assert len(markers) == 26 and n_marks == 156

26 markers = 18 control + 8 check, 156 image marks


## 7. Assemble the `Optimizer`

`add_object` is just `dict_obj[cls_name][id] = obj` — order does not matter, but
*everything* has to be registered, `Point2D`s included, so that
`update_nb_obs_var()` can hand out variable and observation indices.

`opt.set_sigma("Point2D", …)` is the **only** knob that reaches the pixel
residuals; `opt.set_sigma("Image", …)` does not.

Two scop quirks encoded below:

- `_cached_identity_l` (`Optimizer.py:869`) caches the identity-observation values
  and is **never invalidated** — not by `update_nb_obs_var`, not by
  `_invalidate_all_caches`. Pop it before every `fit`, or edits to `img.Xs` are
  silently ignored.
- `opt.check()` (rank / condition diagnostic) hangs above ~2000 variables — it
  runs a sparse `svds(which='SM')` (`Optimizer.py:1476`). Never call it here.


In [31]:
opt = Optimizer()

opt.add_object(cam)
for img in images.values():
    opt.add_object(img)
for pt3 in list(tie_points.values()) + list(markers.values()):
    opt.add_object(pt3)
for pt2 in lst_pt2:
    opt.add_object(pt2)

opt.set_sigma("Point2D", enrich_data.tie_points_sigma_px)  # px, ties AND marks

opt.update_nb_obs_var()
opt.__dict__.pop("_cached_identity_l", None)  # never invalidated

n_identity = len(opt._identity_obs_rows)
expected = 3 * len(images) * USE_CAMERA_PRIORS + 3 * len(data.control_names)
print(f"variables    : {opt.nb_variables}")
print(f"observations : {opt.nb_observations}")
print(f"redundancy   : {opt.nb_observations - opt.nb_variables}")
print(
    f"identity rows: {n_identity}  (3 x {len(images)} camera priors "
    f"+ 3 x {len(data.control_names)} control)"
)

assert n_identity == expected, (n_identity, expected)


  Blocks: 219588 real + 171 identity (total 219759, 171 eliminated)
variables    : 128218
observations : 439347
redundancy   : 311129
identity rows: 171  (3 x 39 camera priors + 3 x 18 control)


## 8. Reprojection before the fit

`Image.compute_batched_residuals(x)` (`Image.py:1045`) returns `(res, idx, …)`
with `res` of shape `(N, 2)` in pixels and `idx` the indices into `img._lst_pt2`,
which is what lets us split ties from markers.

This is the **RMS of the residual norm**; COLMAP's `mean_reprojection_error` is
the *mean* of the norm, so the two differ by ~10 % on identical geometry. The tie
figure should land near **0.82 px** — the Sim3 alignment does not change
reprojection, so this is COLMAP's own post-bundle accuracy seen through scop.


In [32]:
def reproj_rmse(opt, split_id=GCP_ID0):
    # -> (tie RMSE px, n_tie, marker RMSE px, n_marker), RMS of the residual norm
    x = opt.to_jax_vector()
    acc = {False: [0.0, 0], True: [0.0, 0]}  # keyed by "is a marker mark"
    for img in opt.dict_obj["Image"].values():
        out = img.compute_batched_residuals(x)
        if out is None:
            continue
        res, idx, *_ = out
        res = np.asarray(res)
        for k, i in enumerate(idx):
            is_marker = img._lst_pt2[i].get_pt3().get_id() >= split_id
            acc[is_marker][0] += float((res[k] ** 2).sum())
            acc[is_marker][1] += 1
    out = []
    for key in (False, True):
        s, n = acc[key]
        out += [math.sqrt(s / n) if n else float("nan"), n]
    return tuple(out)


pre_tie, n_tie, pre_mrk, n_mrk = reproj_rmse(opt)
print(
    f"pre-fit  tie {pre_tie:.3f} px ({n_tie} obs) | "
    f"marker {pre_mrk:.3f} px ({n_mrk} obs)"
)


pre-fit  tie 0.883 px (219432 obs) | marker 0.356 px (156 obs)


## 9. The adjustment

LM with the Schur complement to get there, then a single Gauss-Newton step with
`compute_Qxx=True` for the covariances.

- **No `robust_kernel`.** `_robust_weights` assumes the identity rows are trailing
  (`dl[:n - n_identity]`, `Optimizer.py:1693`), but here they are interleaved with
  the paired rows, so the weights land on the wrong residuals. The working
  alternative is `opt.disable_outliers(threshold_px=…)` between two `fit` calls.
- Pop `_cached_identity_l` before each call (see above).


In [33]:
t0 = time.time()

opt.__dict__.pop("_cached_identity_l", None)
opt.fit(method="LM", iter_max=25, schur=True, compute_Qxx=False, verbose=True)

opt.__dict__.pop("_cached_identity_l", None)
opt.fit(method="GN", iter_max=1, schur=True, compute_Qxx=True, verbose=True)

post_tie, _, post_mrk, _ = reproj_rmse(opt)
print(f"\nfit in {time.time() - t0:.1f} s")
print(
    f"post-fit tie {post_tie:.3f} px ({n_tie} obs) | "
    f"marker {post_mrk:.3f} px ({n_mrk} obs)"
)


MyOptimizer - JAX-based Fit [LM] (jacobian_mode=sparse, schur)


Levenberg-Marquardt:  24%|▏| 6/25 [00:04<00:15,  1.21iter/s, step_rms=9.809e-04, cost=3.424e+05, λ=7.81e-06, σ₀=0.7417, 



Fit done. Final s0=1.48350 s0_lin=0.74175 ddl=0.62420
MyOptimizer - JAX-based Fit [GN] (jacobian_mode=sparse, schur)


Gauss-Newton: 100%|███| 1/1 [00:00<00:00,  2.73iter/s, step_rms=7.720e-04, cost=3.424e+05, σ₀=0.7417, rms_res=6.242e-01]



Fit done. Final s0=1.48350 s0_lin=0.74175 ddl=0.62420
MyOptimizer - Computing Qxx [Schur diagonal, 128218 vars]
  Schur build: 205ms
  S formation + inversion: 89ms
  Q_pp diagonal (Rust): 741ms
  Qvv (Rust exact): 2100ms
  wi/zi assignment: 267ms
  TOTAL Qxx: 2661ms

fit in 10.4 s
post-fit tie 0.883 px (219432 obs) | marker 0.069 px (156 obs)


## 10. Accuracy — everything read back out of scop

No write-back into the pycolmap `Reconstruction`, no `pycolmap_ba.utils`: every
number below comes from `get_estimated_value` on the scop objects.

With `USE_CAMERA_PRIORS = True` the camera-centre column is a **fit** to the
priors, not an independent accuracy. The **check points** are the only truly
independent figure — they were never observed by the adjustment.


In [34]:
gt_pose = enrich_data.cam_from_world_all(truth)

dc, dr = [], []
for img_id, img in opt.dict_obj["Image"].items():
    name = id2name[img_id]
    est = np.array([img.get_estimated_value(a) for a in ("Xs", "Ys", "Zs")])
    dc.append(est - truth.camera_xyz[name])

    # back to COLMAP axes: R_colmap = diag(1,-1,-1) @ R_scop
    R_est = FLIP @ opk_to_rmat(
        *[img.get_estimated_value(a) for a in ("omega", "phi", "kappa")], np, np
    )
    cos = (np.trace(R_est @ gt_pose[name].rotation.matrix().T) - 1.0) / 2.0
    dr.append(math.degrees(math.acos(float(np.clip(cos, -1.0, 1.0)))))

dc = np.array(dc)
label = "fit to priors" if USE_CAMERA_PRIORS else "TRUE ACCURACY"
print(
    f"camera centres ({label:>13}): bias {np.round(dc.mean(0), 4)} m  "
    f"RMSE {np.sqrt((dc**2).sum(1).mean()):.4f} m"
)
print(
    f"camera rotations              : mean {np.mean(dr):.4f} deg  "
    f"max {np.max(dr):.4f} deg"
)

for group, names, tag in (
    ("control", data.control_names, ""),
    ("check  ", data.check_names, "   <- TRUE ACCURACY"),
):
    e = np.array([
        [markers[n].get_estimated_value(a) for a in "XYZ"] - truth.marker_xyz[n]
        for n in names
    ])
    print(
        f"{group} (n={len(names):2d})              : "
        f"bias {np.round(e.mean(0), 4)} m  "
        f"RMSE {np.sqrt((e**2).sum(1).mean()):.4f} m{tag}"
    )

# The check points must have moved off their surveyed start, otherwise they were
# accidentally constrained and their RMSE is meaningless.
moved = max(
    abs(markers[n].get_estimated_value("X") - float(data.marker_xyz[n][0]))
    for n in data.check_names
)
print(f"\nmax check-point motion off survey: {moved:.4f} m")
assert moved > 1e-6, "check points did not move - they were constrained by mistake"

if REFINE_FOCAL:
    f_est = cam.get_estimated_value("f")
    # get_parameter_std returns the a-priori sigma; scale it by s0 for a-posteriori.
    std = opt.get_parameter_std(cam, "f")
    sigma = float(opt.s0) * std if std is not None else float("nan")
    print(
        f"focal: {f_est:.3f} px +/- {sigma:.3f}  "
        f"(COLMAP {f0:.3f}, nominal {enrich_data.FOCAL_PX:.3f})"
    )


camera centres (fit to priors): bias [ 0.0007 -0.0013  0.0003] m  RMSE 0.0175 m
camera rotations              : mean 0.0000 deg  max 0.0000 deg
control (n=18)              : bias [-0.0001  0.0001 -0.0001] m  RMSE 0.0053 m
check   (n= 8)              : bias [ 0.0012  0.0004 -0.0034] m  RMSE 0.0124 m   <- TRUE ACCURACY

max check-point motion off survey: 0.0047 m
focal: 5865.118 px +/- 0.960  (COLMAP 5865.181, nominal 5865.181)


## 11. scop-native quality

`opt.s0` is the a-posteriori sigma: **≈ 1 means the declared sigmas are right**.
Prefer it to `opt.ddl`, which mixes pixels and metres in one RMS and so is not
interpretable on a mixed problem like this.

`print_report` must be filtered — unfiltered it prints a row per `Point3D`.


In [35]:
print(f"a-posteriori sigma0 : {float(opt.s0):.4f}   (1.0 = declared sigmas are right)")
print(f"tie    RMSE : {pre_tie:.3f} -> {post_tie:.3f} px  ({n_tie} obs)")
print(f"marker RMSE : {pre_mrk:.3f} -> {post_mrk:.3f} px  ({n_mrk} obs)")

opt.print_report(filter_str="Camera")
opt.print_convergence_report()


a-posteriori sigma0 : 1.4835   (1.0 = declared sigmas are right)
tie    RMSE : 0.883 -> 0.883 px  (219432 obs)
marker RMSE : 0.356 -> 0.069 px  (156 obs)
nb_obs=439347 nb_variables=128218 nb_iter=1 s0=1.48350 ddl=0.62420
Camera_1    estimated      value      delta      wi      zi
f           5865.1184  5865.1811    -0.0626    0.00    0.00

                         CONVERGENCE REPORT

Method: GN/GD
Total iterations: 1
Variables: 128218
Observations: 439347
Degrees of freedom: 311129

Final Statistics:───────────────────────────────────────────────────────────────
  Posterior sigma (σ₀): 1.483502
  RMS residuals (ddl): 0.624200

Iteration History:──────────────────────────────────────────────────────────────
  Iter           Cost       Step RMS         σ₀        RMS Res
──────────────────────────────────────────────────────────────
     0   3.423632e+05   7.719710e-04   0.741750   6.242004e-01



In [36]:
# Images only - never call print_report() unfiltered here (one row per Point3D).
opt.print_report(filter_str="Image")


nb_obs=439347 nb_variables=128218 nb_iter=1 s0=1.48350 ddl=0.62420
Image_1     estimated      value      delta      wi      zi
Xs          -193.5142  -193.5067    -0.0075    0.08    0.98
Ys          -159.9865  -160.0000     0.0135    0.14    0.98
Zs           187.9952   188.0000    -0.0048    0.05    0.97
omega         -0.0001    -0.0001     0.0000    0.00    0.00
phi           -0.0000    -0.0000    -0.0000    0.00    0.00
kappa         -1.5708    -1.5708     0.0000    0.00    0.00
Image_2     estimated      value      delta      wi      zi
Xs          -145.7393  -145.7282    -0.0111    0.11    0.98
Ys          -159.9798  -160.0000     0.0202    0.20    0.98
Zs           187.9977   188.0000    -0.0023    0.02    0.97
omega         -0.0001    -0.0001     0.0000    0.00    0.00
phi           -0.0001    -0.0001     0.0000    0.00    0.00
kappa         -1.5708    -1.5708     0.0000    0.00    0.00
Image_3     estimated      value      delta      wi      zi
Xs          -217.3915  -217.3960 

## 12. scop vs pycolmap

Reference run: [`ENRICH-Aerial_pycolmap_sfm_gcp.ipynb`](ENRICH-Aerial_pycolmap_sfm_gcp.ipynb),
same block, same data, COLMAP's Ceres problem.

| | pycolmap + Ceres | scop, 5 000 pts | scop, full block |
|---|---|---|---|
| tie points | 43 749 | 5 000 | 43 749 |
| variables | — | 15 313 | 131 560 |
| **check-point RMSE** | **1.25 cm** | 1.72 cm | **1.21 cm** |
| control RMSE | — | 0.57 cm | 0.52 cm |
| camera centres (fit) | — | 2.75 cm | 1.72 cm |
| focal | 5865.136 px | 5865.098 px | 5865.123 px |
| tie reprojection | 0.817 px (mean) | 0.839 px (RMS) | 0.888 px (RMS) |
| marker reprojection | — | 0.102 px | 0.069 px |
| σ̂₀ | — | 1.28 | 1.50 |
| fit time | — | 3.6 s | 8.1 s |

The full block lands **1.21 cm** on the independent check points against
pycolmap's 1.25 cm — the two engines agree to well under a centimetre on the same
data. σ̂₀ ≈ 1.3–1.5 says the declared sigmas are slightly optimistic (as expected:
one 0.5 px sigma is carrying both ties and marks).

Deliberate deviations, all recorded above where they occur:

1. **One pixel sigma** (0.5 px) for tie points *and* marker marks — scop resolves
   the pixel sigma at class level, so marks cannot be down-weighted separately.
   The pycolmap run used 1.0 px on the marks.
2. **No robust loss** — `robust_kernel` mis-assigns weights when identity rows are
   interleaved with paired rows. Ceres used a Cauchy loss.
3. **LM + Schur** (scop) vs Ceres' trust-region solver.
4. **float32 state** in scop's JAX vector vs double in Ceres.
5. **Subsampled network** by default (5 000 longest tracks); set
   `MIN_TRACK_LENGTH, MAX_POINTS = 2, None` for the full 43 749-point block.

### Next step

Swap the hand-built cells 3–6 for
`load_colmap_reconstruction(SPARSE, max_points=MAX_POINTS, min_track_length=MIN_TRACK_LENGTH)`
and keep cells 6–12 as they are — the loader produces exactly the same objects,
with the same ids and the same conventions.
